In [1]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv(
    "/content/sample_data/LD2011_2014.txt",
    sep=";",              # delimiter
    decimal=",",          # VERY IMPORTANT
    parse_dates=[0],      # first column = datetime
    quotechar='"'         # handles quotes
)

In [ ]:
df.head()

In [ ]:
df.rename(columns={df.columns[0]: "datetime"}, inplace=True)
df.set_index("datetime", inplace=True)

In [ ]:
df.isna().sum()[0]==1

In [ ]:
df.columns

In [ ]:
df["total_load"] = df.sum(axis=1)
df_hourly = df["total_load"].resample("H").sum().reset_index(drop=True)
df_hourly.head()

In [ ]:
# If df_hourly is a Series, convert it to a DataFrame first
if isinstance(df_hourly, pd.Series):
    df_hourly = df_hourly.to_frame(name="total_load")

# Make sure index is datetime
df_hourly.index = pd.to_datetime(df_hourly.index)

# Create a copy so original stays untouched
df_features = df_hourly.copy()

# Basic calendar/time features
df_features["hour_of_day"] = df_features.index.hour
df_features["day_of_week"] = df_features.index.dayofweek   # Monday=0, Sunday=6
df_features["is_weekend"] = (df_features["day_of_week"] >= 5).astype(int)
df_features["month"] = df_features.index.month

# Optional cyclical encoding
df_features["hour_sin"] = np.sin(2 * np.pi * df_features["hour_of_day"] / 24)
df_features["hour_cos"] = np.cos(2 * np.pi * df_features["hour_of_day"] / 24)

df_features["dayofweek_sin"] = np.sin(2 * np.pi * df_features["day_of_week"] / 7)
df_features["dayofweek_cos"] = np.cos(2 * np.pi * df_features["day_of_week"] / 7)

df_features["month_sin"] = np.sin(2 * np.pi * (df_features["month"] - 1) / 12)
df_features["month_cos"] = np.cos(2 * np.pi * (df_features["month"] - 1) / 12)

print(df_features.head())
print(df_features.columns)